In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # points to src/
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [ ]:
columns_to_use = ['V1AF02', 'V1AF14', 'V1AF10','PublicID'] # years of education, family income, English
df1 = pd.read_csv('../../Data/V1A.csv', usecols=columns_to_use,encoding='latin-1')

df2 = pd.read_csv('../../Data/DEMOGRAPHICS.CSV', usecols=['Age_at_V1','PublicID','CRace'],encoding='latin-1')
merged_df = pd.merge(df2, df1, on='PublicID', how='inner')

# Convert the selected demographic and survey columns to integers for modeling.
columns_to_convert = ['Age_at_V1', 'V1AF02', 'V1AF10']
merged_df[columns_to_convert] = merged_df[columns_to_convert].apply(
    pd.to_numeric, errors='coerce'
)

Merged DataFrame shape: (9279, 6)


In [ ]:
# Read the healthcare-response columns that will be collapsed into one flag.
healthcare_columns = ['V1AF15a', 'V1AF15b', 'V1AF15c', 'V1AF15d', 'V1AF15e', 'V1AF15f', 'V1AF15g','PublicID']

# Create a row-level indicator showing whether any response implies healthcare access.
df = pd.read_csv('../../Data/V1A.csv', usecols=healthcare_columns)

df['has_healthcare'] = 0

for index, row in df.iterrows():
    for column in healthcare_columns:
        if row[column] == 1:
            df.at[index, 'has_healthcare'] = 1  # 'Has_Healthcare' 1 if individual has healthcare
            break
        elif row[column] == 0:
            df.at[index, 'has_healthcare'] = 0  # 'Has_Healthcare' 0 if individual does not have healthcare
            break


In [4]:
merged_df = pd.merge(merged_df, df[['PublicID','has_healthcare']], on='PublicID', how='inner')

In [5]:
df_outcomes = pd.read_csv('../Data/modified/Outcome.csv', usecols=['PublicID','MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

merged_df = pd.merge(merged_df, df_outcomes, on='PublicID', how='inner')
merged_df

,PublicID,Age_at_V1,CRace,V1AF02,V1AF10,V1AF14,has_healthcare,MH_outcome
0,00004O,21.0,3.0,4.0,1.0,5,0,1
1,00007I,19.0,1.0,4.0,1.0,4,1,1
2,00008G,30.0,1.0,3.0,1.0,10,0,0
3,00015J,32.0,1.0,7.0,1.0,12,0,0
4,00016H,21.0,2.0,3.0,1.0,7,1,1
...,...,...,...,...,...,...,...,...
7736,17349I,23.0,1.0,4.0,1.0,3,0,1
7737,17350A,22.0,1.0,4.0,1.0,4,1,1
7738,17351V,40.0,1.0,7.0,1.0,11,0,0
7739,17352T,31.0,4.0,8.0,1.0,13,0,1


In [6]:
# Remove non-response codes before modeling.
merged_df = merged_df[merged_df['V1AF14'] != 'D']
merged_df = merged_df[merged_df['V1AF14'] != 'R']

In [7]:

X = merged_df.drop(['MH_outcome', 'PublicID'], axis=1)
y = merged_df['MH_outcome']

train_df = merged_df[merged_df['PublicID'].isin(train_ids)].copy()
test_df = merged_df[merged_df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()

MH_outcome
0    3907
1    2504
Name: count, dtype: int64

In [8]:
numeric_features = ['Age_at_V1', 'V1AF02', 'V1AF10', 'has_healthcare']
categorical_features = ['CRace', 'V1AF14']

# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    numeric_features,
    categorical_features
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.01, 'classifier__l1_ratio': 0.75}
Best Macro F1 Score: 0.5718
Model Coefficients:
num__Age_at_V1: 0.0
num__V1AF02: -0.19511091264461755
num__V1AF10: 0.0
num__has_healthcare: 0.09700551916175286
cat__CRace_1.0: 0.0
cat__CRace_2.0: 0.0
cat__CRace_3.0: 0.0
cat__CRace_4.0: 0.0
cat__CRace_5.0: 0.0
cat__V1AF14_1: 0.0
cat__V1AF14_10: 0.0
cat__V1AF14_11: 0.0
cat__V1AF14_12: 0.0
cat__V1AF14_13: 0.0
cat__V1AF14_14: 0.0
cat__V1AF14_2: 0.0
cat__V1AF14_3: 0.0
cat__V1AF14_4: 0.0
cat__V1AF14_5: 0.0
cat__V1AF14_6: 0.0
cat__V1AF14_7: 0.0
cat__V1AF14_8: 0.0
cat__V1AF14_9: 0.0
Evaluation Metrics for LR with shared preprocessing and macro F1 grid search:
Accuracy: 0.5964
Precision: 0.5756
Recall: 0.5760
F1-score: 0.5758
ROC AUC: 0.5854


In [9]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    numeric_features,
    categorical_features
)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 15, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 700}
Best Macro F1 Score: 0.5578
Feature Importances:
num__Age_at_V1: 0.32732898957652296
num__V1AF02: 0.22097338050915194
num__V1AF10: 0.025349519948293046
num__has_healthcare: 0.05331173532521639
cat__CRace_1.0: 0.03325144500030387
cat__CRace_2.0: 0.022196966457532854
cat__CRace_3.0: 0.029682510557978502
cat__CRace_4.0: 0.012558040596402744
cat__CRace_5.0: 0.01812095630196652
cat__V1AF14_1: 0.024566376643404178
cat__V1AF14_10: 0.01697123342240623
cat__V1AF14_11: 0.021299990399040087
cat__V1AF14_12: 0.027990811954845877
cat__V1AF14_13: 0.012207070118039991
cat__V1AF14_14: 0.019015497416231538
cat__V1AF14_2: 0.018697484738219243
cat__V1AF14_3: 0.027141814188457864
cat__V1AF14_4: 0.011077074961363495
cat__V1AF14_5: 0.017158231630977225
cat__V1AF14_6: 0.013224849120185928
cat__

In [10]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    numeric_features,
    categorical_features
)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.01, 'classifier__max_depth': 4, 'classifier__n_estimators': 80, 'classifier__subsample': 0.5}
Best Macro F1 Score: 0.5713
Feature Importances:
num__Age_at_V1: 0.038633111864328384
num__V1AF02: 0.10598304122686386
num__V1AF10: 0.039348799735307693
num__has_healthcare: 0.1178833469748497
cat__CRace_1.0: 0.04333028569817543
cat__CRace_2.0: 0.02878277748823166
cat__CRace_3.0: 0.03546285629272461
cat__CRace_4.0: 0.028410902246832848
cat__CRace_5.0: 0.027462903410196304
cat__V1AF14_1: 0.04894191026687622
cat__V1AF14_10: 0.03537772595882416
cat__V1AF14_11: 0.04916202649474144
cat__V1AF14_12: 0.05463901162147522
cat__V1AF14_13: 0.01846366748213768
cat__V1AF14_14: 0.04780798405408859
cat__V1AF14_2: 0.03178388252854347
cat__V1AF14_3: 0.04727837070822716
cat__V1AF14_4: 0.033310383558273315
cat__V1AF14_5: 0.03566891700029373
cat__V1AF14_6: 0.02

In [12]:
# Run the SVM pipeline with macro F1 grid search. (If output hidden, inspect the file itself with the "less" command)
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    numeric_features,
    categorical_features
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 0.1, 'classifier__estimator__gamma': 'auto', 'classifier__estimator__kernel': 'rbf'}
Best Macro F1 Score: 0.5751
Evaluation Metrics for SVM with shared preprocessing and macro F1 grid search:
Accuracy: 0.6074
Precision: 0.5778
Recall: 0.5723
F1-score: 0.5728
ROC AUC: 0.5958
